# ClonField Studio · Colab
Escolha um runtime com GPU e execute na ordem. Cadastre `CIVITAI_TOKEN` e `HF_TOKEN` no painel Secrets do Colab e autorize acesso. O código vem do GitHub e os arquivos autenticados são guardados no runtime ou na pasta do Drive escolhida.

O Studio abre com downloads de modelos sob demanda. A célula de pré-download do Civitai é opcional e começa com a lista vazia.

Para investigar o encerramento do Qwen, marque `PROBE_QWEN` na última célula e execute. Ela roda uma inferência mínima, guarda a imagem quando concluir e registra a última etapa e a memória. Quando quiser abrir a interface, desmarque `PROBE_QWEN` e rode a célula novamente.


In [ ]:
# Atualizar o código público
from pathlib import Path
import subprocess, sys

origin = "https://github.com/Vit0lze/clonfield-studio.git"
repo = Path("/content/clonfield_runtime/clonfield-studio")
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    current_origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if current_origin != origin:
        raise RuntimeError("O diretório existente aponta para outro repositório.")
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", origin, str(repo)], check=True)


In [ ]:
# 1. Instalar dependências. Versões compatíveis do Gradio e Pillow.
import subprocess, sys
from importlib import metadata
pillow_spec = 'Pillow==11.3.0'
packages = [
    'git+https://github.com/huggingface/diffusers.git', 'transformers>=5.17',
    'accelerate', 'bitsandbytes', 'peft', 'safetensors', 'gradio==5.49.1', 'huggingface_hub',
    pillow_spec, 'requests', 'spandrel', 'numpy'
]
print('Instalando dependências; Pillow:', pillow_spec, flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', *packages])
print('Dependências instaladas. Continue com as próximas células.', flush=True)


In [ ]:
# Credenciais opcionais do Colab. Não escreva tokens no notebook.
# Crie Secrets chamados HF_TOKEN e CIVITAI_TOKEN no painel lateral e permita acesso.
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
for secret_name in ('HF_TOKEN', 'CIVITAI_TOKEN'):
    value = None
    if userdata is not None:
        try:
            value = userdata.get(secret_name)
        except Exception:
            value = None
    if value:
        os.environ[secret_name] = value
        print(secret_name + ': configurado para esta sessão')
    elif os.environ.get(secret_name):
        print(secret_name + ': já disponível no ambiente')
    else:
        print(secret_name + ': não configurado (downloads públicos continuam disponíveis)')
    value = None


In [ ]:
# 2. Opcional: montar Drive para persistir LoRAs, modelos e resultados
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['CLONFIELD_DIR'] = '/content/drive/MyDrive/ClonField_Lab'
else:
    import os
    os.environ['CLONFIELD_DIR'] = '/content/ClonField_Lab'
print('Pasta:', os.environ['CLONFIELD_DIR'])


In [ ]:
# Opcional: pré-baixar apenas versões Civitai escolhidas
# Deixe [] para abrir o Studio direto. Para usar, informe IDs de versão do assets.json.
VERSOES_CIVITAI = []
if VERSOES_CIVITAI:
    args = [sys.executable, "-u", str(repo / "cf_assets.py")]
    for version in VERSOES_CIVITAI:
        args.extend(["--version", str(version)])
    if subprocess.run(args).returncode:
        print("Alguns downloads ficaram pendentes. Você pode continuar para abrir o Studio.")
else:
    print("Pré-download pulado. Continue para iniciar o worker e o Studio.")


In [ ]:
# Iniciar o worker local de upscale. Os pesos são baixados no primeiro job.
START_COMFYUI = True
PREPARE_OPENJUTSU = False

if START_COMFYUI:
    import json
    import os
    import socket
    import subprocess
    import sys
    import time
    import urllib.error
    import urllib.request
    import venv
    from pathlib import Path

    revision = "v0.44.0"
    runtime = Path("/content/clonfield_runtime")
    checkout = runtime / f"ComfyUI-{revision}"
    python_env = runtime / f"comfy_venv-{revision}"
    models = runtime / "ComfyUI" / "models"
    log_path = runtime / "comfyui.log"
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))

    def comfy_has_seedvr2():
        try:
            for name in ("SeedVR2Preprocess", "SeedVR2TemporalChunk", "SeedVR2TemporalMerge"):
                with opener.open(f"http://127.0.0.1:8188/object_info/{name}", timeout=2) as response:
                    if name not in json.load(response):
                        return False
            return True
        except (OSError, ValueError, urllib.error.URLError):
            return False

    if comfy_has_seedvr2():
        if PREPARE_OPENJUTSU:
            raise RuntimeError("Pare o worker antes de instalar o pack OpenJutsu.")
        os.environ.setdefault("CLONFIELD_COMFY_MODELS", str(models))
        print("ComfyUI com SeedVR2 já está ativo.")
    else:
        with socket.socket() as port_check:
            port_check.settimeout(1)
            if port_check.connect_ex(("127.0.0.1", 8188)) == 0:
                raise RuntimeError("A porta 8188 tem um worker antigo. Reinicia a sessão do Colab e roda esta versão.")
        models.mkdir(parents=True, exist_ok=True)
        if not checkout.exists():
            subprocess.run(["git", "init", str(checkout)], check=True)
            subprocess.run(["git", "-C", str(checkout), "remote", "add", "origin",
                            "https://github.com/Comfy-Org/ComfyUI.git"], check=True)
        current = subprocess.run(["git", "-C", str(checkout), "rev-parse", "HEAD"],
                                 capture_output=True, text=True)
        if current.returncode:
            subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", "tag", revision], check=True)
            subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            pinned_commit = subprocess.check_output(
                ["git", "-C", str(checkout), "rev-parse", f"{revision}^{{commit}}"], text=True
            ).strip()
            if current.stdout.strip() != pinned_commit:
                raise RuntimeError(f"O checkout dedicado diverge da versão {revision}.")

        executable = python_env / "bin" / "python"
        marker = python_env / "comfy_revision.txt"
        if not executable.is_file() or not marker.is_file() or marker.read_text().strip() != revision:
            print("Preparando o ambiente Python do ComfyUI...", flush=True)
            # Recria a configuração se a instalação anterior parou no ensurepip.
            venv.EnvBuilder(with_pip=False, system_site_packages=True).create(python_env)
            subprocess.run(
                [sys.executable, "-m", "pip", "--python", str(executable),
                 "install", "-r", str(checkout / "requirements.txt")],
                check=True,
            )
            marker.write_text(revision, encoding="utf-8")
        os.environ["CLONFIELD_COMFY_MODELS"] = str(models)
        if PREPARE_OPENJUTSU:
            subprocess.run([sys.executable, str(repo / "prepare_openjutsu.py"), "--checkout", str(checkout),
                            "--models", str(models), "--python", str(executable)], check=True)
        with log_path.open("w") as log:
            comfy_process = subprocess.Popen(
                [str(executable), "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram",
                 "--models-directory", str(models)],
                cwd=checkout, stdout=log, stderr=subprocess.STDOUT,
            )
        for _ in range(90):
            if comfy_has_seedvr2():
                print(f"ComfyUI pronto. Logs em {log_path}")
                break
            if comfy_process.poll() is not None:
                raise RuntimeError(log_path.read_text()[-5000:])
            time.sleep(1)
        else:
            comfy_process.terminate()
            raise RuntimeError("ComfyUI não ficou pronto em 90 s. " + log_path.read_text()[-3000:])


In [ ]:
# Iniciar Studio ou reproduzir a geração mínima com diagnóstico
PROBE_QWEN = False
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_path = repo.parent / "logs" / f"diagnostico-{stamp}.jsonl"
args = [sys.executable, "-u", str(repo / "cf_diagnose.py"), "--log", str(log_path)]
if PROBE_QWEN:
    args.append("--probe")
print(f"Diagnóstico desta execução: {log_path}", flush=True)
status = subprocess.run(args).returncode
if status:
    raise RuntimeError(f"O Studio encerrou com código {status}. Consulte o diagnóstico em {log_path}.")
